In [1]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain langchain-community langchain-google-genai
%pip install -Uq langchain_chroma
%pip install -Uq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --upgrade -q langchain-google-genai google-generativeai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [19]:
import json
from typing import List

from google import genai
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage

PREFERRED_CHAT_MODEL = "gemini-2.5-flash"
CHAT_MODEL_FALLBACKS = ["gemini-2.0-flash", "gemini-1.5-flash"]


def resolve_chat_model() -> str:
    """Use preferred model when available; otherwise pick a safe Flash fallback."""
    try:
        client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
        available_models = {
            model.name.split("/")[-1]
            for model in client.models.list()
            if "generateContent" in getattr(model, "supported_actions", [])
        }
    except Exception as exc:
        print(f"⚠️ Could not list models: {exc}")
        available_models = set()

    candidates = [PREFERRED_CHAT_MODEL, *CHAT_MODEL_FALLBACKS]
    for model_name in candidates:
        if not available_models or model_name in available_models:
            if model_name != PREFERRED_CHAT_MODEL:
                print(f"⚠️ {PREFERRED_CHAT_MODEL} unavailable, using {model_name}")
            return model_name

    return PREFERRED_CHAT_MODEL

In [20]:
def partition_document(file_path: str):
    print(f"📄 Partitioning document: {file_path}")
  
    elements = partition_pdf(
        filename=file_path,
        strategy="fast", 
        infer_table_structure=False, # Table structure usually requires hi_res
        extract_image_block_types=[], # No OCR needed
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

In [21]:
def create_chunks_by_title(elements):
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

In [22]:
def separate_content_types(chunk):
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            elif element_type == 'Image':
                if hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

In [23]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    try:
        llm = ChatGoogleGenerativeAI(
            model=resolve_chat_model(),
            temperature=0
        )

        prompt_text = f"""
You are creating a searchable description for document retrieval.

TEXT:
{text}
"""

        if tables:
            prompt_text += "\nTABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        prompt_text += """
Generate a detailed, searchable summary including:
- Key facts
- Topics
- Questions it answers
- Insights from tables/images
"""

        response = llm.invoke(prompt_text)
        return response.content

    except Exception as e:
        print(f"❌ Gemini summary failed: {e}")
        return text[:300]

In [24]:
def summarise_chunks(chunks):
    print("🧠 Processing chunks...")
    
    docs = []
    
    for chunk in chunks:
        content_data = separate_content_types(chunk)
        
        if content_data['tables'] or content_data['images']:
            enhanced = create_ai_enhanced_summary(
                content_data['text'],
                content_data['tables'],
                content_data['images']
            )
        else:
            enhanced = content_data['text']
        
        doc = Document(
            page_content=enhanced,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        docs.append(doc)
    
    print(f"✅ Processed {len(docs)} chunks")
    return docs

In [25]:
def create_vector_store(documents, persist_directory="chroma_db"):
    print("🔮 Creating vector store...")
    
    # We use the 'legacy' string which is actually the most stable across all versions
    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001"
    )
    
    db = Chroma.from_documents(
        documents,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    
    print("✅ Vector DB ready")
    return db

In [26]:
def run_pipeline(pdf_path):
    elements = partition_document(pdf_path)
    chunks = create_chunks_by_title(elements)
    docs = summarise_chunks(chunks)
    db = create_vector_store(docs)
    return db

db = run_pipeline(r"d:\Users\User\Downloads\attention_is_all_you_need.pdf")

📄 Partitioning document: d:\Users\User\Downloads\attention_is_all_you_need.pdf


No languages specified, defaulting to English.


✅ Extracted 393 elements
🔨 Creating smart chunks...
✅ Created 33 chunks
🧠 Processing chunks...
✅ Processed 33 chunks
🔮 Creating vector store...
✅ Vector DB ready


In [27]:
def generate_final_answer(chunks, query):
    llm = ChatGoogleGenerativeAI(
        model=resolve_chat_model(),
        temperature=0
    )

    context = ""
    for i, chunk in enumerate(chunks):
        context += f"\n--- Doc {i+1} ---\n{chunk.page_content}\n"

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)
    return response.content

In [28]:
query = "What are the components of Transformer?"
retriever = db.as_retriever(search_kwargs={"k": 3})

chunks = retriever.invoke(query)

answer = generate_final_answer(chunks, query)
print(answer)

The Transformer's overall architecture uses stacked self-attention and point-wise, fully connected layers for both its encoder and decoder.

Specifically:
*   **Encoder:** Composed of a stack of N = 6 identical layers. Each layer has two sub-layers:
    *   A multi-head self-attention mechanism.
    *   A simple, position-wise fully connected feed-forward network.
    *   Residual connections and layer normalization are employed around each sub-layer.
*   **Decoder:** Also composed of a stack of N = 6 identical layers. In addition to the two sub-layers found in each encoder layer, the decoder inserts a third sub-layer:
    *   A multi-head attention mechanism that operates over the output of the encoder stack.
    *   Similar to the encoder, residual connections and layer normalization are used around each sub-layer.
    *   The self-attention sub-layer in the decoder is modified with masking to prevent attending to subsequent positions.
